# Procesamiento

#### Libraries

Currently Using

- Pandas
- Numpy
- Os


In [81]:
import os
import traceback
import pandas as pd
from pandasgui import show
from IPython.display import display

pd.set_option("display.max_columns", None)

### Numero de archivos y Nombres de los deportes

In [82]:
target_folder = "Datos_disciplinas"

# Contar el número de documentos en la carpeta
def count_number(target_folder):
    try:
        file_count = sum(1 for entry in os.scandir(target_folder) if entry.is_file())
        print(f"Total de documentos: {file_count}")

    except FileNotFoundError:
        print(f"Error: no se encontró la carpeta '{target_folder}'.")


# Contar y nombrar la cantidad de deportes
def ramas_d():
    sports = []
    files = []

    print("Lista de deportes:")

    for file in os.listdir(target_folder):
        if file.endswith(".xlsx"):
            files.append(file)

            name = file.rsplit(".", 1)[0]
            sports.append(name)

            print(f"{len(sports)}: {name}")

    return sports, files

### Checar Concentimiento y Atletas en total


In [ ]:
def concent(df):
    si_concent = df[~df["C000 ID"].isna()]
    if not si_concent.empty:
        display(si_concent)
    else:
        print("No hay concent en el DataFrame.")

    #display(si_concent)

def count_athletes(df):
    unique_athletes = df["Athlete"].nunique()
    display(unique_athletes.value_counts())


### Aux


In [84]:

def preguntar_sn(mensaje):
    while True:
        r = input(mensaje).strip().lower()
        if r in ("s", "si", "sí"):
            return True
        if r in ("n", "no"):
            return False
        print("Introduce s o n.")


def pedir_nombres(validos):
    while True:
        texto = input("Nombres exactos separados por comas (Enter para cancelar): ")
        nombres = [n.strip() for n in texto.split(",") if n.strip()]
        if not nombres:
            return None
        #Validar los nombres en la lista
        no_existen = [n for n in nombres if n not in validos]
        if not no_existen:
            return nombres
        print("No están en la lista:", ", ".join(no_existen))


def pedir_saltos(validos):
    while True:
        texto = input(
            "Números de los saltos a eliminar, separados por comas "
            "(Enter para cancelar): "
        )
        partes = [p.strip() for p in texto.split(",") if p.strip()]
        if not partes:
            return None
        try:
            numeros = [int(p) for p in partes]
        except ValueError:
            print("Solo se permiten números enteros.")
            continue
        invalidos = [n for n in numeros if n not in validos]
        if not invalidos:
            return numeros
        print("No pertenecen a los atletas seleccionados:", invalidos)




### Jump Count


In [ ]:
def jump_count(df, sheet_name):

    df_tmp = df.copy()

    # El índice identifica cada salto, así que debe ser único
    if not df_tmp.index.is_unique:
        df_tmp = df_tmp.reset_index(drop=True)

    columnas_base = [
        "Athlete",
        "Test Type",
        "Trial",
        "Jump Height (Imp-Mom) [cm]",
        "Concentric Impulse (Left) [N s]",
        "Concentric Impulse (Right) [N s]",
        "Concentric RFD (Left) [N/s]",
        "Concentric RFD (Right) [N/s]",
        "Eccentric Braking RFD (Left) [N/s]",
        "Eccentric Braking RFD (Right) [N/s]",
        "RSI-modified (Imp-Mom) [m/s]",
        "Takeoff Peak Force (Right) [N]",
        "Takeoff Peak Force (Left) [N]",
        "Concentric Time to Peak Force (Left) [ms]",
        "Concentric Time to Peak Force (Right) [ms]",
        "Force at Peak Power (Left) [N]",
        "Force at Peak Power (Right) [N]",
        "Peak Landing Force (Left) [N]",
        "Peak Landing Force (Right) [N]",
        "Landing RFD (Right) [N/s]",
        "Landing RFD (Left) [N/s]",
        "Concentric Peak Force (Left) [N]",
        "Concentric Peak Force (Right) [N]",
        "Eccentric Peak Force (Left) [N]",
        "Eccentric Peak Force (Right) [N]",
    ]

    # Se filtran las columnas que existen en el DataFrame
    cols_mostrar = [c for c in columnas_base if c in df_tmp.columns]

    while True:
        # Se cuentan los intentos de cada atleta
        conteo = df_tmp["Athlete"].value_counts().reset_index()
        conteo.columns = ["Athlete", "Trials"]
        reprobados = conteo[conteo["Trials"] != 3]
    
        if reprobados.empty:
            print(f"No hay atletas con un número de intentos distinto de 3 "
                  f"en la sesión '{sheet_name}'.")
            return df_tmp

        #Aqui se filtran los atletas que no tienen 3 intentos y se muestran sus saltos
        df_reprobados = df_tmp.loc[
            df_tmp["Athlete"].isin(reprobados["Athlete"]), cols_mostrar
        ].sort_values("Athlete")

        print(f"\n=== Sesión '{sheet_name}' ===")
        print("Atletas con intentos distintos de 3:")
        display(reprobados)
        print("Sus saltos:")
        display(df_reprobados)

        while True:
            accion = input(
                "\n¿Qué quieres hacer?\n"
                "  [1] Eliminar a todos los atletas de la lista\n"
                "  [2] Revisar atletas específicos\n"
                "  [3] Terminar sin eliminar más\n"
                "Opción: "
            ).strip()
            if accion in ("1", "2", "3"):
                break
            print("Introduce 1, 2 o 3. No hay atletas con un número de intentos distinto de 3 en la sesión 'Sesion1'")

        if accion == "1":
            df_tmp = df_tmp.loc[
                ~df_tmp["Athlete"].isin(reprobados["Athlete"])
            ].copy()
            print("Se eliminaron los atletas de la lista.")
            return df_tmp

        if accion == "3":
            return df_tmp

        nombres = pedir_nombres(set(reprobados["Athlete"]))
        if nombres is None:
            continue

        saltos = df_tmp.loc[df_tmp["Athlete"].isin(nombres), cols_mostrar]
        print("\nSaltos de los atletas seleccionados:")
        display(saltos)

        numeros = pedir_saltos(set(saltos.index))
        if numeros is None:
            continue

        print("\nSe eliminarán estos saltos:")
        display(df_tmp.loc[numeros, cols_mostrar])

        if preguntar_sn(f"¿Eliminar {len(numeros)} salto(s)? (s/n): "):
            df_tmp = df_tmp.drop(index=numeros)
            print("Saltos eliminados.")
        else:
            print("No se eliminó nada.")

        if not preguntar_sn("¿Seguir revisando? (s/n): "):
            return df_tmp

### Main


In [86]:
def main():
    count_number(target_folder)
    sports, files = ramas_d()

    sport_input = input("Ingrese el numero del deporte a procesar: ")

    if not (sport_input.isdigit() and 1 <= int(sport_input) <= len(files)):
        print("Deporte no encontrado.")
        return {}

    file_path = os.path.join(target_folder, files[int(sport_input) - 1])
    xl = pd.ExcelFile(file_path)
    print(f"Sesiones: {xl.sheet_names}")

    sesiones_finales = {}

    for sheet_name in xl.sheet_names:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=8)
            print("aqui: ")
            concent(df)
            sesiones_finales[sheet_name] = jump_count(df, sheet_name)

        except Exception as e:
            print(f"  Error al procesar la hoja '{sheet_name}': {e}")
            traceback.print_exc()

    return sesiones_finales


sesiones_finales = main()

Total de documentos: 20
Lista de deportes:
1: Americano
2: AtletismoSalto_Femenil
3: AtletismoSalto_Varonil
4: AtletismoVelocidad_Femenil
5: AtletismoVelocidad_Varonil
6: Basketball_Femenil
7: Basketball_Varonil
8: Natacion_Femenil
9: Natacion_Varonil
10: Soccer_Femenil
11: Soccer_Varonil
12: TaeKwonDo_Femenil
13: TaeKwonDo_Varonil
14: TenisdeMesa_Femenil
15: TenisdeMesa_Varonil
16: Tennis_Femenil
17: Tennis_Varonil
18: Tocho_Femenil
19: Volleyball_Femenil
20: Volleyball_Varonil
Sesiones: ['Sesion1']
aqui: 


,Forms ID,C000 ID,Athlete,Date of Birth,Género,Disciplina,Test Type,Test Date,Body Weight [kg],Trial,Jump Height (Imp-Mom) [cm],Concentric Impulse (Left) [N s],Concentric Impulse (Right) [N s],Concentric RFD (Left) [N/s],Concentric RFD (Right) [N/s],Eccentric Braking RFD (Left) [N/s],Eccentric Braking RFD (Right) [N/s],RSI-modified (Imp-Mom) [m/s],Takeoff Peak Force (Right) [N],Takeoff Peak Force (Left) [N],Concentric Time to Peak Force (Left) [ms],Concentric Time to Peak Force (Right) [ms],Force at Peak Power (Left) [N],Force at Peak Power (Right) [N],Peak Landing Force (Left) [N],Peak Landing Force (Right) [N],Landing RFD (Right) [N/s],Landing RFD (Left) [N/s],Concentric Peak Force (Left) [N],Concentric Peak Force (Right) [N],Eccentric Peak Force (Left) [N],Eccentric Peak Force (Right) [N]
3,208,C131,Judith Silva,17/10/2006,Mujer,Tenis de mesa,Countermovement Jump,2026-08-31 11:29:26.220,75.37,Trial 1,17.1,197.1,212.1,0,1500,1895,1969,0.20,758,733,0,2,595,628,1244,2134,33342,16373,726,752,733,758
4,208,C131,Judith Silva,17/10/2006,Mujer,Tenis de mesa,Countermovement Jump,2026-08-31 11:29:26.220,75.37,Trial 2,18.3,196.4,203.9,0,0,2067,2386,0.21,846,744,2,0,612,621,1768,1583,26381,25262,737,830,744,846
5,208,C131,Judith Silva,17/10/2006,Mujer,Tenis de mesa,Countermovement Jump,2026-08-31 11:29:26.220,75.37,Trial 3,18.2,207.1,207.6,0,0,2082,2407,0.20,848,760,0,0,609,593,2135,2024,32643,33365,750,824,760,848
6,207,C130,Valeria DeGasperin,24/08/2006,Mujer,Tenis de mesa,Countermovement Jump,2026-08-31 11:22:23.143,65.92,Trial 1,25.7,162.1,168.0,1000,0,1981,1828,0.31,721,651,8,0,615,621,1826,1439,24808,35122,651,720,648,721
7,207,C130,Valeria DeGasperin,24/08/2006,Mujer,Tenis de mesa,Countermovement Jump,2026-08-31 11:22:23.143,65.92,Trial 2,25.6,158.1,170.0,1000,0,2192,2090,0.33,745,662,194,2,632,655,2176,1891,37817,45341,662,745,656,745
8,207,C130,Valeria DeGasperin,24/08/2006,Mujer,Tenis de mesa,Countermovement Jump,2026-08-31 11:22:23.143,65.92,Trial 3,25.8,161.6,171.3,750,1250,2062,1979,0.35,738,659,2,4,618,622,2009,2222,41146,35882,659,738,658,735


No hay atletas con un número de intentos distinto de 3 en la sesión 'Sesion1'.


In [87]:
#show(**sesiones_finales)                                # todas las sesiones en PandasGUI
#todo = pd.concat(sesiones_finales, names=["Sesion"])    # un solo DataFrame para analizar